# AI助手项目笔记本

本笔记本提供了AI助手项目的概述，包括LLM组件和虚拟桌面宠物组件。


## 项目概述

AI助手项目由两个主要组件组成：
1. **LLM（大语言模型）** - 一个基于AMD ROCm优化的3B参数对话语言模型
2. **虚拟桌面宠物** - 一个集成了人格模拟和WSL2命令执行能力的AI桌面宠物系统


## LLM组件

### 模型架构

- **模型规模**: 3B参数
- **架构**: Decoder-only Transformer (类GPT)
- **隐藏维度**: 3072
- **层数**: 32
- **注意力头**: 24 (GQA: 8个KV头)
- **中间层维度**: 8192
- **最大序列长度**: 4096
- **词表大小**: 65536


In [ ]:
# 安装LLM依赖
!pip install -r llm/requirements.txt

# 安装ROCm版本的PyTorch
!pip install torch --index-url https://download.pytorch.org/whl/rocm7.1

In [ ]:
# 导入LLM组件
import sys
import os

# 添加LLM目录到路径
sys.path.append(os.path.abspath('llm'))

from configs.model_config import ModelConfig, InferenceConfig
from core.model import LLMForCausalLM
from inference.generator import SamplingGenerator
from transformers import AutoTokenizer

In [ ]:
# 初始化分词器和模型
tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese')

# 创建模型配置
model_config = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id if tokenizer.bos_token_id else 1,
    eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else 2
)

# 创建模型
model = LLMForCausalLM(model_config)

# 创建推理配置
inference_config = InferenceConfig(
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id or 2
)

# 创建生成器
generator = SamplingGenerator(
    model=model,
    tokenizer=tokenizer,
    config=inference_config
)

In [ ]:
# 测试LLM生成
prompt = '你好，最近怎么样？'
response = generator.generate(prompt)
print(f'Prompt: {prompt}')
print(f'Response: {response[len(prompt):].strip()}')

## 虚拟桌面宠物

### 系统架构

- **客户端-服务器架构**: 分离的客户端和服务器组件
- **客户端**: 提供用户界面，处理用户输入，显示AI回复
- **服务器**: 处理客户端请求，执行LLM推理，管理知识库


In [ ]:
# 启动虚拟桌面宠物服务器
# 注意：这需要在单独的终端中运行
# !cd 'Virtual Desktop Companion/server' && python main.py

In [ ]:
# 启动虚拟桌面宠物本地服务器
# 注意：这需要在单独的终端中运行
# !cd 'Virtual Desktop Companion/client' && node local-server.js

## 使用示例

### 示例1: 基本对话

1. 在浏览器中打开 `Virtual Desktop Companion/client/index.html`
2. 在聊天框中输入消息
3. AI桌面宠物将根据其人格做出回应


### 示例2: WSL2命令执行

1. 在聊天框中输入命令，例如：'列出当前目录中的文件'
2. 系统将生成WSL2命令
3. 本地服务器将执行命令
4. 结果将显示在聊天界面中


### 示例3: 知识检索

1. 询问关于WSL2的问题，例如：'如何安装WSL2？'
2. RAG系统将检索相关信息
3. AI将基于检索到的知识提供全面的答案


## 技术亮点

1. **AMD ROCm优化**: 利用AMD GPU能力实现高效的模型训练和推理
2. **FlashAttention 2**: 优化的注意力机制，提高内存使用效率
3. **RAG技术**: 通过检索增强生成提升知识能力
4. **多API集成**: 结合不同的LLM API以获得最佳性能
5. **本地命令执行**: 通过本地服务器安全执行WSL2命令
6. **人格模拟**: 丰富的人格和情绪状态，提供引人入胜的交互


## 未来发展

1. **知识库扩展**: 覆盖更多技术领域和主题
2. **人格增强**: 深化人格模拟和情绪智能
3. **安全优化**: 提高命令执行的安全性和效率
4. **跨平台支持**: 扩展到更多操作系统和环境
5. **个性化选项**: 为用户提供更多定制功能


## 结论

AI助手项目展示了先进的LLM技术与实际应用的集成，创建了一个综合的AI助手系统，结合了人格模拟、知识检索和命令执行能力。通过利用AMD ROCm优化，系统在提供引人入胜和有用的用户交互的同时，实现了高效的性能。
